### Import libraries

In [1]:
import importlib
from snowflake.snowpark.functions import col,trim,split,lit
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
import sys 
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np

In [7]:
def add_fiscal_year(df, date_col_name, target_col_name="FY"):
    c = F.col(date_col_name)
    
    return df.with_column(
        target_col_name,
        F.when(c.between(F.lit("2022-04-01"), F.lit("2023-03-31")), F.lit("FY'23"))
         .when(c.between(F.lit("2023-04-01"), F.lit("2024-03-31")), F.lit("FY'24"))
         .when(c.between(F.lit("2024-04-01"), F.lit("2025-03-31")), F.lit("FY'25"))
         .when(c.between(F.lit("2025-04-01"), F.lit("2026-03-31")), F.lit("FY'26"))
         .when(c.between(F.lit("2026-04-01"), F.lit("2027-03-31")), F.lit("FY'27"))
         .otherwise(F.lit("Unknown"))
    )

def aggregate_and_sort_by_fy(df, group_col="FY", sum_col="NET_SALES", target_sum_name="TOTAL_NET_SALES"):
    return (
        df.group_by(F.col(group_col))
        .agg(F.sum(F.col(sum_col)).alias(target_sum_name))
        .sort(F.split_part(F.col(group_col), F.lit("'"), F.lit(2)).asc())
    )

In [2]:
session = Session.builder.configs(snowflake_conn_prop).create()

### Step 1 - Execute stored procedure to get the latest data

In [3]:
result = session.sql("CALL MOP_DATABASE.SOQ.TFT_DATA_PIPELINE_STEP1('2022-04-01','2027-04-01')").collect()

print(result[0][0])

Pipeline execution complete for interval 2022-04-01 -> 2027-04-01. Log uploaded to Stage.


In [4]:
snowpark_df = session.table('MOP_DATABASE.SOQ.TRAIN_AND_TEST_DATA_FOR_TFT')

In [5]:
snowflake_utils.shape_of_snowpark_df(snowpark_df)

(7726321, 61)

In [8]:
snowpark_df_new = add_fiscal_year(df=snowpark_df,date_col_name='MONTH_OF_SALE',target_col_name='FY')
agg_final_data_sales = aggregate_and_sort_by_fy(df=snowpark_df_new,group_col="FY",sum_col="NET_SALES",target_sum_name="TOTAL_NET_SALES")
agg_final_data_sales.show()


-------------------------------
|"FY"     |"TOTAL_NET_SALES"  |
-------------------------------
|Unknown  |0.000000           |
|FY'23    |4687708.000000     |
|FY'24    |5136645.000000     |
|FY'25    |5256582.000000     |
|FY'26    |5836980.000000     |
|FY'27    |1340977.000000     |
-------------------------------



### Step 2 - Adding dealer city, city category, and zonal office name to the data

In [9]:
dealer_city_and_city_cat = session.sql("""
WITH CTE AS 
(SELECT TRIM(SPLIT_PART(PARENT_DEALER_NAME,'-',1)) AS PARENT_DEALER_CODE,DEALER_CITY,X_CITY_CATEGORY,ZONAL_OFFICE_NAME
FROM ANALYTICS_DATABASE.ANALYTICS.FACT_CUSTOMER_PROFILE
QUALIFY ROW_NUMBER() OVER(PARTITION BY TRIM(SPLIT_PART(PARENT_DEALER_NAME,'-',1)) ORDER BY FIRST_SALE_DT DESC)=1)
(SELECT DISTINCT PARENT_DEALER_CODE,DEALER_CITY,X_CITY_CATEGORY,ZONAL_OFFICE_NAME
FROM CTE)
""")

In [10]:
dealer_city_and_city_cat.select(F.col("PARENT_DEALER_CODE")).print_schema()

root
 |-- "PARENT_DEALER_CODE": StringType(100) (nullable = True)


In [11]:
snowpark_df = snowpark_df.with_column('PARENT_DEALER_CODE',F.trim(F.col("PARENT_DEALER_CODE")))

In [12]:
#Add dealer city and other columns
merged_df = snowpark_df.join(dealer_city_and_city_cat,on="PARENT_DEALER_CODE",how="left")

snowflake_utils.shape_of_snowpark_df(merged_df)

(7726321, 64)

### Step - 3 : Data manipulation

In [13]:
cols_to_check = ["PARENT_DEALER_CODE", "DEALER_CITY", "ZONAL_OFFICE_NAME"]

# Build a list of aggregate expressions
null_counts = [
    _sum(when(is_null(col(c)), 1).otherwise(0)).alias(f"NULL_{c}") 
    for c in cols_to_check
]

# Run the aggregation
merged_df.select(null_counts).show()

-----------------------------------------------------------------------------
|"NULL_PARENT_DEALER_CODE"  |"NULL_DEALER_CITY"  |"NULL_ZONAL_OFFICE_NAME"  |
-----------------------------------------------------------------------------
|0                          |244                 |244                       |
-----------------------------------------------------------------------------



In [14]:
merged_df=merged_df.na.fill({"DEALER_CITY":"Unknown","ZONAL_OFFICE_NAME":"Unknown","X_CITY_CATEGORY":"Unknown"})

In [15]:
#Convert to pandas
pandas_df = merged_df.to_pandas()

In [16]:
null_columns = pandas_df.columns[pandas_df.isnull().sum()>0]
null_columns

Index(['DUSSEHRA_(VIJAYADASHAMI)_DAYS', 'NUM_DAYS_MONTH'], dtype='object')

In [17]:
pandas_df.fillna(value=0,inplace=True)

In [18]:
#Remove MODEL_NAME,MODEL_FAMILY_CODE
pandas_df.drop(columns=['MODEL_NAME','MODEL_FAMILY_CODE','YEAR'],inplace=True)

pandas_df.rename(columns={'NUM_DAYS_MONTH':'NUM_FESTIVE_DAYS_MONTH'},inplace=True)

In [19]:
pandas_df['MONTH_OF_SALE'] = pd.to_datetime(pandas_df['MONTH_OF_SALE'])

pandas_df.set_index('MONTH_OF_SALE',inplace=True)

In [20]:
# 1. Reset index so we can check it
df_reset = pandas_df.reset_index()

# 2. Identify duplicates across the three columns
duplicates = df_reset[df_reset.duplicated(subset=['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE'], keep=False)]

# 3. Show the problematic rows
if not duplicates.empty:
    print("Duplicate combinations found:")
    print(duplicates.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).head(10))

### Step 4 - Save the data

In [21]:
pandas_df.to_csv(r"Validated_data_for_TFT_training.csv")

# pandas_df.to_csv(r"April_2022_to_Mar_2023_data.csv")